# 01 Inspect WLASL1000 Dataset

## Purpose
This notebook prepares WLASL1000 for model development. It checks the WLASL1000 metadata, verifies which videos exist locally, creates a usable video index, and saves a label map.

## Why this matters
WLASL metadata can contain missing videos. Later extraction/training notebooks should only use videos that actually exist in `data/raw/ASL/videos`.

## Expected inputs

```text
data/raw/ASL/WLASL1000/nslt_1000.json
data/raw/ASL/WLASL1000/wlasl_class_list.txt
data/raw/ASL/videos/
```

In [ ]:
from pathlib import Path
import json
import pandas as pd

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL1000"
PREFIX = "wlasl1000"

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ASL" / DATASET_NAME
VIDEOS_DIR = PROJECT_ROOT / "data" / "raw" / "ASL" / "videos"

META_FILE = RAW_DIR / "nslt_1000.json"
CLASS_LIST_FILE = RAW_DIR / "wlasl_class_list.txt"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
LABEL_MAP_DIR = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_INDEX_FILE = PROCESSED_DIR / f"{PREFIX}_video_index.csv"
LABEL_MAP_FILE = LABEL_MAP_DIR / f"asl_{PREFIX}_labels.json"

print("Metadata exists:", META_FILE.exists(), META_FILE)
print("Class list exists:", CLASS_LIST_FILE.exists(), CLASS_LIST_FILE)
print("Shared videos folder exists:", VIDEOS_DIR.exists(), VIDEOS_DIR)

## 1. Load class list and metadata

In [ ]:
class_names = {}
with open(CLASS_LIST_FILE, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            class_names[int(parts[0])] = parts[1]

with open(META_FILE, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Class names loaded:", len(class_names))
print("Metadata entries:", len(metadata))

first_id = list(metadata.keys())[0]
print("Example video ID:", first_id)
print("Example item:", metadata[first_id])

## 2. Build video availability table

In [ ]:
records = []

for video_id, item in metadata.items():
    original_class_id = item["action"][0]
    gloss = class_names.get(original_class_id, "unknown")
    video_path = VIDEOS_DIR / f"{video_id}.mp4"

    records.append({
        "video_id": video_id,
        "original_class_id": original_class_id,
        "gloss": gloss,
        "video_path": str(video_path),
        "exists": video_path.exists()
    })

df = pd.DataFrame(records)

print("Total metadata videos:", len(df))
print("Existing videos:", df["exists"].sum())
print("Missing videos:", (~df["exists"]).sum())
print("Metadata classes:", df["original_class_id"].nunique())

df.head()

## 3. Create usable index and label map

In [ ]:
df_usable = df[df["exists"] == True].copy()

glosses = sorted(df_usable["gloss"].unique())
gloss_to_label_id = {gloss: idx for idx, gloss in enumerate(glosses)}
df_usable["label_id"] = df_usable["gloss"].map(gloss_to_label_id)

df_usable.to_csv(VIDEO_INDEX_FILE, index=False)

label_map = {}
for _, row in df_usable.drop_duplicates("label_id").iterrows():
    label_id = int(row["label_id"])
    label_map[label_id] = {
        "language": "ASL",
        "dataset": DATASET_NAME,
        "gloss": row["gloss"],
        "display_text": row["gloss"],
        "original_class_id": int(row["original_class_id"])
    }

label_map = dict(sorted(label_map.items(), key=lambda x: x[0]))

with open(LABEL_MAP_FILE, "w", encoding="utf-8") as f:
    json.dump(label_map, f, indent=4)

print("Saved video index:", VIDEO_INDEX_FILE)
print("Saved label map:", LABEL_MAP_FILE)
print("Usable videos:", len(df_usable))
print("Usable classes:", df_usable["label_id"].nunique())
df_usable.head()

## 4. Review class distribution

In [ ]:
class_counts = df_usable["gloss"].value_counts()

print("Class distribution summary")
print("--------------------------")
print("Classes:", len(class_counts))
print("Minimum videos per class:", class_counts.min())
print("Maximum videos per class:", class_counts.max())
print("Average videos per class:", round(class_counts.mean(), 2))
print("\nClasses with fewer than 5 usable videos:")
print(class_counts[class_counts < 5])

class_counts.head(30)

## Next step

Continue with:

```text
02_extract_wlasl1000_keypoints.ipynb
```